In [1]:
#import libraries
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Make the repo root importable so `from src.data_loader import load_cases` works
# regardless of where Jupyter was launched from.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_cases

In [2]:
# load the full merged dataset (both shards) via the shared loading pipeline.
# First run builds+caches data/processed/all_cases.parquet; later runs just read the cache.
df = load_cases()
print(df.shape)

(33610, 20)


In [3]:
# sanity check: columns and case_type breakdown (hearing + appeal, both shards)
print(df.columns)
print(df["case_type"].value_counts())

Index(['row_id', 'case_number', 'date', 'outcome', 'guidelines', 'summary',
       'full_text', 'sor_allegations', 'mitigating_factors', 'judge',
       'source_url', 'formal_findings', 'case_type', 'appeal_board_members',
       'who_appealed', 'judges_findings_of_fact', 'judges_analysis',
       'discussion', 'order', 'source_shard'],
      dtype='str')
case_type
hearing    27973
appeal      5637
Name: count, dtype: int64


In [4]:
# clean the dataset (removes incomplete cases, randomizes the cases) and select 2000 cases
df = df.dropna(subset=['full_text', 'outcome']).sample(frac=1, random_state=42)
df = df.head(2000)

# Drop outcome classes that are too rare *within this 2000-row sample* to stratify on.
# REVOKED has only ~15 cases across the full 33.6k-row dataset (vs. 0 in shard 1 alone,
# which is all Izzy's original version ever saw), so a fixed-size sample can easily draw
# just 1 example of it -- too few for stratified train/val split below.
MIN_CLASS_COUNT = 2
outcome_counts = df['outcome'].value_counts()
rare_outcomes = outcome_counts[outcome_counts < MIN_CLASS_COUNT].index.tolist()
if rare_outcomes:
    print(f"Dropping rare outcome classes from this sample (< {MIN_CLASS_COUNT} cases): {rare_outcomes}")
    df = df[~df['outcome'].isin(rare_outcomes)]

print(df.shape)
print(df['outcome'].value_counts())

Dropping rare outcome classes from this sample (< 2 cases): ['REVOKED']
(1999, 20)
outcome
DENIED      1263
GRANTED      536
UNKNOWN      174
REMANDED      26
Name: count, dtype: int64


In [5]:
# split the text into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    df['full_text'], 
    df['outcome'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['outcome']
)

print("Training cases:", len(X_train))
print("Validation cases:", len(X_val))

Training cases: 1599
Validation cases: 400


In [6]:
# create a TF-IDF vectorizer
vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))

Xtr = vec.fit_transform(X_train) #x training data
Xv = vec.transform(X_val)   # x validation data

print("Training TF-IDF shape:", Xtr.shape)
print("Validation TF-IDF shape:", Xv.shape)

Training TF-IDF shape: (1599, 2000)
Validation TF-IDF shape: (400, 2000)
